In [ ]:
import numpy as np
from adaptive_latents import datasets
from sim_stim import make_srs
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
import pathlib
import matplotlib.pyplot as plt
from adaptive_latents.plotting_functions import AnimationManager
from adaptive_latents.stim_designer import StimDesigner, OptimizationMethod
from tqdm.autonotebook import tqdm
from IPython.display import display, clear_output



In [ ]:
rng = np.random.default_rng(0)
d = datasets.Zong22Dataset()
data = d.neural_data

srs = make_srs(data, rng, comparison_preset='visualization', n_runs=1, show_tqdm=True)


i= 40
sr = srs['learning from stim'][0]

fig, axs = plt.subplots(ncols=2, figsize=(10,4), sharex=False, sharey=False, layout='constrained')

latents = sr.log['latents'].slice_by_time(slice(30,None))
axs[0].plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
stim_s = sr.log['stim_intended_samples'].t - latents.dt

l = 1
r = 5.1
ax_n = 0
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = axs[ax_n].plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
axs[ax_n].plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')

for arrow_index in [17, 50]:
    axs[0].annotate('',
                    xytext=(latents[arrow_index, 0], latents[arrow_index, 1]),
                    xy=(latents[arrow_index+1, 0], latents[arrow_index+1, 1]),
                    arrowprops=dict(arrowstyle="simple", color='C0'),
                    size=11
                    )


u = sr.stim_designer.log[i]['u']
idx = np.argsort(np.abs(u))[::-1]
print(np.linalg.norm(u,ord=0))

high_d = sr.log['high_d_with_stim'].slice_by_time(slice(center_t-l,center_t+r))
axs[1].plot(high_d.t, high_d[:,idx[:int(np.linalg.norm(u,ord=0))]]);
for stim_t in stim_s:
    axs[1].axvline(stim_t, color='r')


In [ ]:
fig, ax = plt.subplots()


latents = sr.log['latents'].slice_by_time(slice(30,None))
ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')

i = 40
l = 1
r = 5.1
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = ax.plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')


In [ ]:
from matplotlib.animation import FFMpegWriter, PillowWriter, HTMLWriter, ImageMagickWriter

latents = sr.log['latents'].slice_by_time(slice(30,None))


# am = AnimationManager(filename_stem='stim_video', outdir='.', filetype='gif', dpi=400)


fig, axs = plt.subplots(1, 1, figsize=(10,10), layout='constrained', squeeze=False)

# movie_writer = FFMpegWriter(codec='libvpx-vp9', fps=20, bitrate=-1)
movie_writer = ImageMagickWriter(fps=20, bitrate=-1)
movie_writer.setup(fig, 'zong_stim.gif', dpi=100)

fig.patch.set_alpha(0.)
for t in tqdm(np.linspace(300, 311, 200+1)):
    ax = axs[0,0]
    ax.cla()
    ax.patch.set_alpha(0.)

    ax.plot(latents[:, 0], latents[:, 1], alpha=.5, color='k', lw=1)

    sl = slice(t-2,t)
    sub_latents = latents.slice_by_time(sl)
    ax.plot(sub_latents[:, 0], sub_latents[:, 1], color='k', lw=6)

    stim_s = sr.log['stim_intended_samples'].slice_by_time(sl).t - latents.dt
    latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
    ax.scatter(latents_s[:, 0], latents_s[:, 1], s=75, color='r', zorder=1000)


    movie_writer.grab_frame()

movie_writer.finish()

In [ ]:
fig, ax = plt.subplots()

ax.matshow(d.ops['meanImg'], cmap='gray')
xs, ys = list(zip(*[cell['med'] for cell in d.stat]))




u = sr.stim_designer.log[i]['u']
v = sr.stim_designer.log[i]['v']
equivalent_projection_matrix = sr.stim_designer.log[i]['equiv_proj_mat']
u_to_s_function=lambda u: 10 * equivalent_projection_matrix.T @ u
new_u, l = sr.stim_designer.design_stim_jaxopt(v, u.size, u_to_s_function=u_to_s_function)




In [ ]:

stim_designer = StimDesigner(should_log=True)


In [ ]:
fig, axs = plt.subplots(figsize=(10,5), ncols=2, layout='constrained')
ax = axs[1]
latents = sr.log['latents'].slice_by_time(slice(30,None))
ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')



i = 40
l = 1
r = 0
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = ax.plot(latents[:-1, 0], latents[:-1, 1], color='k')
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='g')



u = sr.stim_designer.log[i]['u']
v = sr.stim_designer.log[i]['v']

v = 0 * v
theta = np.linspace(0, 2 * np.pi, 20)[9]
v[0,0] = np.cos(theta)
v[1,0] = np.sin(theta)

equivalent_projection_matrix = sr.stim_designer.log[i]['equiv_proj_mat']
u_to_s_function=lambda u: equivalent_projection_matrix.T @ u



stim_designer.optimization_method = OptimizationMethod.JAXOPT
new_u = stim_designer.design_stim(v=v, u_dimension=u.size, u_to_s_function=u_to_s_function, equivalent_projection_matrix=equivalent_projection_matrix)


s = u_to_s_function(new_u)
s_marker = axs[1].annotate('',
                      xytext=(latents_s[0,0], latents_s[0,1]),
                      xy=(latents_s[0,0]+s[0], latents_s[0,1]+s[1]),
                      arrowprops=dict(arrowstyle="-|>", color='g'),
                      size=12,
                      ).arrow_patch


v_arrow = ax.annotate('',
                xytext=(latents_s[0,0], latents_s[0,1]),
                xy=(latents_s[0,0]+v[0,0], latents_s[0,1]+v[1,0]),
                arrowprops=dict(arrowstyle="-|>", color='r'),
                size=12
                )



def make_legend_arrow(legend, orig_handle,
                      xdescent, ydescent,
                      width, height, fontsize):
    p = mpatches.FancyArrow(width, 0.5*height, -width, 0, length_includes_head=True, head_width=0.7*height, head_length=.23*width)
    return p

# ax.legend(*zip(
#     (v_arrow.arrow_patch, 'v'),
#     (s_marker, 's (predicted)'),
# ), handler_map={mpatches.FancyArrowPatch : HandlerPatch(patch_func=make_legend_arrow),})


ax = axs[0]
ax.matshow(d.ops['meanImg'], cmap='gray')
# fig.colorbar([0,1])

c = np.array(new_u)
xs, ys = list(zip(*[cell['med'] for cell in d.stat]))
c[c == 0] = np.nan
cmap = ax.scatter(ys, xs, s=14, c=c, vmin=0, vmax=1, cmap='plasma')


fig.colorbar(cmap)



## for _ in tqdm(range(1000)):
##     stim_designer.optimization_method = OptimizationMethod.JAXOPT
##     v = stim_designer.desired_stim_direction(equivalent_projection_matrix=equivalent_projection_matrix, stim_direction_type='random', rng=stim_designer.rng)
##     new_u = stim_designer.design_stim(v=v, u_dimension=u.size, u_to_s_function=u_to_s_function, equivalent_projection_matrix=equivalent_projection_matrix)
##
##     s = u_to_s_function(new_u)
##     s_marker = axs[1].scatter(latents_s[0,0]+s[0], latents_s[0,1]+s[1], color='g', s=3, alpha=.5)



axs[0].axis('off');
axs[1].axis('off');


